In [13]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/churn.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
df.head()

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,class
0,16,128.0,415,0,1,25.0,265.1,110.0,45.07,197.4,99.0,16.78,244.7,91.0,11.01,10.0,3.0,2.70,1,0
1,35,107.0,415,0,1,26.0,161.6,123.0,27.47,195.5,103.0,16.62,254.4,103.0,11.45,13.7,3.0,3.70,1,0
2,31,137.0,415,0,0,0.0,243.4,114.0,41.38,121.2,110.0,10.30,162.6,104.0,7.32,12.2,5.0,3.29,0,0
3,35,84.0,408,1,0,0.0,299.4,71.0,50.90,61.9,88.0,5.26,196.9,89.0,8.86,6.6,7.0,1.78,2,0
4,36,75.0,415,1,0,0.0,166.7,113.0,28.34,148.3,122.0,12.61,186.9,121.0,8.41,10.1,3.0,2.73,3,0


In [15]:
df.dtypes

state                             object
account_length                   float64
area_code                         object
international_plan                object
voice_mail_plan                   object
number_vmail_messages            float64
total_day_minutes                float64
total_day_calls                  float64
total_day_charge                 float64
total_eve_minutes                float64
total_eve_calls                  float64
total_eve_charge                 float64
total_night_minutes              float64
total_night_calls                float64
total_night_charge               float64
total_intl_minutes               float64
total_intl_calls                 float64
total_intl_charge                float64
number_customer_service_calls     object
class                             object
dtype: object

In [17]:
df['class'] = df['class'].astype(float)
df['number_customer_service_calls'] = df['number_customer_service_calls'].astype(float)
df['international_plan'] = df['international_plan'].astype(int)
df['voice_mail_plan'] = df['voice_mail_plan'].astype(int)

In [ ]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['state', 'area_code' ]

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "Churn",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, Churn, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_churn_empirical.csv", index=True)
print("Done!")

Empirical, TelcoChurn, LogReg, MinHash, 1, 1, 0.8043742104872067, 0.4770382457252326, 0.869, 0.36097560975609755, 0.578125, 0.2624113475177305, 0.10793304443359375
Empirical, TelcoChurn, LogReg, MinHash, 1, 2, 0.8271617169890769, 0.4776980358390039, 0.866, 0.33, 0.559322033898305, 0.23404255319148937, 0.09617733955383301
Empirical, TelcoChurn, LogReg, MinHash, 1, 3, 0.8296633888985213, 0.4644162659296679, 0.865, 0.3283582089552239, 0.55, 0.23404255319148937, 0.07096171379089355
Empirical, TelcoChurn, LogReg, MinHash, 1, 4, 0.8382497783906234, 0.4974339196847473, 0.866, 0.3163265306122449, 0.5740740740740741, 0.21830985915492956, 0.06841325759887695
Empirical, TelcoChurn, LogReg, MinHash, 1, 5, 0.8147591844774944, 0.41643174732075217, 0.857, 0.2513089005235602, 0.4897959183673469, 0.16901408450704225, 0.06966018676757812
Empirical, TelcoChurn, LogReg, MinHash, 2, 1, 0.848496107134306, 0.49594743058112206, 0.875, 0.37185929648241206, 0.6379310344827587, 0.2624113475177305, 0.070455312728

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CART, Leave-one-out, 4, 4, 0.6879083357956597, 0.35946675401244677, 0.888, 0.5087719298245614, 0.6744186046511628, 0.4084507042253521, 0.062436819076538086
Empirical, TelcoChurn, CART, Leave-one-out, 4, 5, 0.7923027676548803, 0.5815389864962974, 0.931, 0.7112970711297071, 0.8762886597938144, 0.5985915492957746, 0.0576481819152832
Empirical, TelcoChurn, CART, Leave-one-out, 5, 1, 0.637769466392556, 0.3424365474573714, 0.892, 0.425531914893617, 0.851063829787234, 0.28368794326241137, 0.0589604377746582
Empirical, TelcoChurn, CART, Leave-one-out, 5, 2, 0.7016529198556791, 0.42488810086682427, 0.905, 0.5539906103286385, 0.8194444444444444, 0.41843971631205673, 0.05918169021606445


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CART, Leave-one-out, 5, 3, 0.5, 0.141, 0.859, 0.0, 0.0, 0.0, 0.05077695846557617
Empirical, TelcoChurn, CART, Leave-one-out, 5, 4, 0.7846777635510029, 0.5643550469483569, 0.928, 0.6974789915966386, 0.8645833333333334, 0.5845070422535211, 0.0719614028930664
Empirical, TelcoChurn, CART, Leave-one-out, 5, 5, 0.6203256837059653, 0.32266901408450704, 0.888, 0.38461538461538464, 0.875, 0.24647887323943662, 0.0711207389831543
Empirical, TelcoChurn, CART, K-fold, 1, 1, 0.851600492078039, 0.530594149985312, 0.908, 0.7032258064516129, 0.6449704142011834, 0.7730496453900709, 0.08633160591125488
Empirical, TelcoChurn, CART, K-fold, 1, 2, 0.8390013127585266, 0.5623137166138525, 0.922, 0.723404255319149, 0.723404255319149, 0.723404255319149, 0.0869290828704834
Empirical, TelcoChurn, CART, K-fold, 1, 3, 0.8561497370354775, 0.5863713031311816, 0.926, 0.7430555555555556, 0.7278911564625851, 0.7588652482269503, 0.08012723922729492
Empirical, TelcoChurn, CART, K-fold, 1, 4, 0.81702

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (187 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CatBoost, Leave-one-out, 5, 4, 0.8890886109195968, 0.6361040495226389, 0.858, 0.0, 0.0, 0.0, 2.7679507732391357


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CatBoost, Leave-one-out, 5, 5, 0.850725565514298, 0.6627286529508813, 0.858, 0.0, 0.0, 0.0, 2.687487840652466
Empirical, TelcoChurn, CatBoost, K-fold, 1, 1, 0.9305724122557155, 0.897751996380597, 0.962, 0.8538461538461538, 0.9327731092436975, 0.7872340425531915, 2.987131118774414
Empirical, TelcoChurn, CatBoost, K-fold, 1, 2, 0.9407524830951377, 0.9064454346048832, 0.969, 0.8803088803088803, 0.9661016949152542, 0.8085106382978723, 2.9611146450042725
Empirical, TelcoChurn, CatBoost, K-fold, 1, 3, 0.946119106003187, 0.9086364354517857, 0.966, 0.8712121212121212, 0.9349593495934959, 0.8156028368794326, 2.9798526763916016
Empirical, TelcoChurn, CatBoost, K-fold, 1, 4, 0.9029350930759381, 0.8595517348223399, 0.956, 0.8253968253968254, 0.9454545454545454, 0.7323943661971831, 2.965226173400879
Empirical, TelcoChurn, CatBoost, K-fold, 1, 5, 0.8988722545060572, 0.8535799284880511, 0.956, 0.824, 0.9537037037037037, 0.7253521126760564, 3.0076887607574463
Empirical, TelcoChu